In [ ]:
from math import log
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA

from cryoclassifier import CryoFormerLite
from dataset import CryoEMDataset

plt.rcParams["image.cmap"] = "gray"

rng = np.random.default_rng()


def load_model(model_path, num_classes, img_size=128, device="cpu"):
    model = CryoFormerLite(num_classes=num_classes, img_size=img_size)
    model = model.to(device)
    ckpt = torch.load(model_path, map_location=device)

    if isinstance(ckpt, dict) and "model" in ckpt:
        state_dict = ckpt["model"]
    elif isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=True)
    model.eval()

    return model

def extract_last_layer_embeddings(model, images, device):
    with torch.no_grad():
        x = model.bandpass(images.to(device))
        x = model.stem(x)
        x = model.stage1(x)
        x = model.down1(x)
        x = model.stage2(x)
        x = model.down2(x)
        x = model.radial_attn(x)
        x = model.stage3(x)
        x = model.down3(x)
        x = model.stage4(x)
        x = x.mean(dim=(-2, -1))
        x = model.head_norm(x)
    return x.cpu()

In [ ]:
model_path = "cryo_model.pth"

particles_df = pd.read_csv("predictions.csv")
particles_ds = CryoEMDataset(
    proj_dir=Path("<cryosparc_dir>"), df=particles_df, train=False, crop_size=128
)

num_classes = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the model
model = load_model(model_path, num_classes, img_size=128, device=device)
model.eval()

## Calculate Embeddings

In [ ]:
col = "predicted_particle"

In [ ]:
sample_per_class = 300
class_names = sorted(particles_df[col].unique())
sample_indices = []
sample_labels = []

for class_name in class_names:
    class_indices = particles_df.index[particles_df[col] == class_name].to_numpy()
    take = min(sample_per_class, len(class_indices))
    chosen = rng.choice(class_indices, size=take, replace=False)
    sample_indices.extend(chosen.tolist())
    sample_labels.extend([class_name] * take)

batch_size = 32
embeddings = []

for start in range(0, len(sample_indices), batch_size):
    batch_indices = sample_indices[start : start + batch_size]
    batch_images = torch.stack([particles_ds[i] for i in batch_indices], dim=0)
    embeddings.append(extract_last_layer_embeddings(model, batch_images, device))

embeddings = torch.cat(embeddings, dim=0).numpy()

In [ ]:

class_names = sorted(particles_df[col].unique())
sample_labels = np.array(sample_labels)

embedding_2d = PCA(n_components=2).fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(4, 4))
palette = plt.cm.tab10(np.linspace(0, 1, len(class_names)))

for color, class_name in zip(palette, class_names):
    mask = sample_labels == class_name
    ax.scatter(
        embedding_2d[mask, 0],
        embedding_2d[mask, 1],
        s=18,
        alpha=0.75,
        color=color,
        label=class_name,
    )

ax.set_title("CryoFormerLite\npenultimate-layer embedding (PCA to 2D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()